# Install Dependencies

In [ ]:
!pip install -q openai

# Load Modules

In [ ]:
import pandas as pd
import json
from openai import OpenAI
from sklearn.model_selection import train_test_split
import re

client = OpenAI(api_key='API_KEY')

# Define Functions

In [ ]:
def format_prompt(data):

    """

    Background: This function helps format data into a list of dicts into the required shape for fine tuning

    Params:
    data (list): list of dicts

    Returns:
    training_data_list (list): a list in the proper format for converting to jsonl

    """

    training_data_list = []

    for x in data:

        updated_data = {
            "messages": [
                {
                    "role": "system",
                    "content": x['system_message']
                },
                {
                    "role": "user",
                    "content": x['user_content']
                }
            ]
        }

        training_data_list.append(updated_data)

    print(training_data_list)

    return training_data_list

In [ ]:
def convert_to_jsonl_and_save(data_list, filename):

    """
    Background:
    This function converts the data_list provided into a jsonl file

    Params:
    data_list (list): a list of a dict ready to convert to jsonl
    filename (str): the name of the filename we want to convert

    """

    with open(filename, 'w') as file:
        for data_dict in data_list:
            json_str = json.dumps(data_dict)  # Convert dictionary to JSON string
            file.write(json_str + '\n')  # Write to file with a newline

    print(f"✅ Data successfully written to {filename}")

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 20
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("predictions.csv")

# Load Model

In [ ]:
model = 'gpt-4.1-2025-04-14'

# Load Data

In [ ]:
data = pd.read_excel('sample_math.xlsx')

# Zero Shot

## Prepare Prompt

In [ ]:
data['system_message'] = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer
'''

data['user_content']= None
for i, text in enumerate(data['Question']):
    prompt = f'''The question you have to answer:
    Question:
    {text}
    Options:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    Answer:'''

    data.at[i, 'user_content'] = prompt

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('QA-Math-ZeroShot-gpt4-1.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
zero_shot_data = format_prompt(datadict)

In [ ]:
# Save as jsonl
convert_to_jsonl_and_save(zero_shot_data, 'NC-ZeroShot.jsonl')

✅ Data successfully written to NC-ZeroShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= zero_shot_data[0]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'a) 36'

In [ ]:
y_true = data2['Answer Key'].values

In [ ]:
zs = pd.DataFrame()

zs['Question'] = data2['Question']
zs['Label'] = y_true
zs = zs.reset_index(drop=True)
zs.head()

,Question,Label
0,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,A
1,جد ناتج مايلي : 6 × 3981\n,C
2,عدد من منزلتین مجموعهما = ١٠,A
3,من أدوات القياس,C
4,أي من المسائل التالية ناتج جمعها يساوي 16,A


In [ ]:
store_predictions(zs, model, zero_shot_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('QA-Math-GPT41-ZeroShot-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
d) nan,5
b) 2,4
b) 7,3
a) 12,3
c) 6,3
...,...
c) 900,1
d) 878,1
a) وتر,1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
pred['Prediction'].shape[0]

200

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
  if "a)" in answer:
    preds.append("A")
  elif "b)" in answer:
    preds.append("B")
  elif "c)" in answer:
    preds.append("C")
  elif "d)" in answer:
    preds.append("D")
  else:
    preds.append("Unclassified")

In [ ]:
np.unique(preds)

array(['A', 'B', 'C', 'D'], dtype='<U1')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['A', 'B', 'C', 'D']
mapping = {'A':0, 'B':1, 'C':2, 'D':3}

def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(zs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=labels)
print('\nClassification Report:')
print(class_report)

Accuracy: 0.815

f1_score:  0.8140342723854336
\precision:  0.8216812354312354
ecall:  0.815

Classification Report:
              precision    recall  f1-score   support

           A     0.8056    0.8923    0.8467        65
           B     0.8269    0.8958    0.8600        48
           C     0.8889    0.7164    0.7934        67
           D     0.6364    0.7000    0.6667        20

    accuracy                         0.8150       200
   macro avg     0.7894    0.8011    0.7917       200
weighted avg     0.8217    0.8150    0.8140       200



# Few Shot

## Prepare Prompt

In [ ]:
data['system_message'] = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

Example 1
Question:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

Options:
a) 220 أوم
b) 202 أوم
c) 100 أوم
d) 484 أوم

Answer:
d) 484 أوم

Example 2
Question:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

Options:
a) جلوكوزيدات
b) مستقبلات
c) أحماض أمينية غير بروتينية
d) سيفالوسبورين

Answer:
b) مستقبلات

Example 3
Question:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

Options:
a) ناتج الطرح
b) المطروح
c) المطروح منه
d) لاشيء مما سبق

Answer:
a) ناتج الطرح
'''

data['user_content']= None
for i, text in enumerate(data['Question']):
    prompt = f'''The question you have to answer:
    Question:
    {text}
    Options:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    Answer:'''

    data.at[i, 'user_content'] = prompt

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('QA-Math-FewShot-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
few_shot_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(few_shot_data, 'NC-FewShot.jsonl')

✅ Data successfully written to NC-FewShot.jsonl


## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= few_shot_data[1]['messages'],
      temperature = 0.2,
      max_tokens= 20
  )

response.choices[0].message.content

'a) 23886'

In [ ]:
y_true = data2['Answer Key'].values

In [ ]:
fs = pd.DataFrame()

fs['Question'] = data2['Question']
fs['Label'] = y_true
fs = fs.reset_index(drop=True)
fs.head()

,Question,Label
0,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,A
1,جد ناتج مايلي : 6 × 3981\n,C
2,عدد من منزلتین مجموعهما = ١٠,A
3,من أدوات القياس,C
4,أي من المسائل التالية ناتج جمعها يساوي 16,A


In [ ]:
store_predictions(fs, model, few_shot_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('QA-Math-gpt41-FewShot-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
d) nan,6
b) 2,4
a) 12,3
c) 6,3
a) 0,2
...,...
a) 15624,1
d) 878,1
a) وتر,1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
  if "a)" in answer:
    preds.append("A")
  elif "b)" in answer:
    preds.append("B")
  elif "c)" in answer:
    preds.append("C")
  elif "d)" in answer:
    preds.append("D")
  else:
    preds.append("Unclassified")

In [ ]:
np.unique(preds)

array(['A', 'B', 'C', 'D'], dtype='<U1')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['A', 'B', 'C', 'D']
mapping = {'A':0, 'B':1, 'C':2, 'D':3}

def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(fs.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=labels)
print('\nClassification Report:')
print(class_report)

Accuracy: 0.805

f1_score:  0.8052113865932047
\precision:  0.8141514041514042
ecall:  0.805

Classification Report:
              precision    recall  f1-score   support

           A     0.8143    0.8769    0.8444        65
           B     0.7885    0.8542    0.8200        48
           C     0.8889    0.7164    0.7934        67
           D     0.6250    0.7500    0.6818        20

    accuracy                         0.8050       200
   macro avg     0.7792    0.7994    0.7849       200
weighted avg     0.8142    0.8050    0.8052       200



# CoT

## Prepare Prompt

In [ ]:
data['system_message'] = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

Follow these reasoning steps to determine the correct answer:

Step 1: Understand the question
Carefully read and comprehend the question. Identify what is being asked, including the underlying need and the context.

Step 2: Analyze the options
Carefully read each of the four options. Consider the meaning of each option and how it relates to the question.

Step 3: Evaluate the options
Apply your knowledge and reasoning to assess each option. Eliminate clearly incorrect options. Compare the remaining options to determine which is most accurate.

Step 4: Select the best answer
Based on your evaluation in the previous steps, choose the single most logical and correct option.

Step 5: Output the answer
Write the correct answer in this exact format:
a/b/c/d) Answer

Example 1
Question:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

Options:
a) 220 أوم
b) 202 أوم
c) 100 أوم
d) 484 أوم

Thinking:
- Step 1: Understand the question
We are asked to calculate the electrical resistance of a lamp rated at 220 volts and 100 watts.
- Step 2: Analyze the options
The resistance must be computed based on the given voltage and power. The options are 220 ohms, 202 ohms, 100 ohms, and 484 ohms.
- Step 3: Evaluate the options
We know the formula for electrical resistance in terms of voltage and power is:
R = V^2/P
Substitute the given values:
R = 220^2/100 = 48400/100 = 484 ohms
- Step 4: Select the correct answer
From the options, 484 أوم corresponds to option d.

Final Answer:
d) 484 أوم

Example 2
Question:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

Options:
a) جلوكوزيدات
b) مستقبلات
c) أحماض أمينية غير بروتينية
d) سيفالوسبورين

Thinking:
- Step 1: Understand the question
We are asked: Which of these functions similarly to white blood cells in humans?
White blood cells (WBCs) are part of the immune system — they fight infections by detecting and neutralizing pathogens (bacteria, viruses, etc.).
- Step 2: Analyze the options
a) Glycosides (جلوكوزيدات): These are sugar-based compounds, not immune-related.
b) Receptors (مستقبلات): Receptors help in detecting and recognizing substances (including pathogens), playing a role in immunity and signaling.
c) Non-protein amino acids (أحماض أمينية غير بروتينية): These are rare metabolic intermediates, not directly involved in immune function.
d) Cephalosporins (سيفالوسبورين): These are antibiotics; they kill bacteria but are not a cellular component of the immune system.
- Step 3: Evaluate the options
Which of these functions similarly to WBCs — that is, by detecting or responding to pathogens?
Receptors help the immune system recognize invaders — similar to how WBCs detect and respond to pathogens.
- Step 4: Select the correct answer
The option that most closely mimics the immune detection function of WBCs is receptors (مستقبلات).

Final Answer:
b) مستقبلات

Example 3
Question:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

Options:
a) ناتج الطرح
b) المطروح
c) المطروح منه
d) لاشيء مما سبق

Thinking:
- Step 1: Understand the question
We are asked: What is the name of the number 50 in the subtraction operation 70 – 20 = 50?
In subtraction:
The first number (70) is called the minuend (المطروح منه).
The second number (20) is called the subtrahend (المطروح).
The result (50) is called the difference (ناتج الطرح).
- Step 2: Analyze the options
a) ناتج الطرح: "Result of subtraction" — this is the correct mathematical term for the outcome of the subtraction (difference).
b) المطروح: "The number being subtracted" — this is 20, not 50.
c) المطروح منه: "The number from which another number is subtracted" — this is 70, not 50.
d) لاشيء مما سبق: "None of the above" — not applicable since option (a) is correct.
- Step 3: Evaluate the options
The number 50 is the result of the subtraction, also known as the difference.
- Step 4: Select the correct answer
The correct answer is ناتج الطرح.

Final Answer:
a) ناتج الطرح
'''

data['user_content']= None
for i, text in enumerate(data['Question']):
    prompt = f'''The question you have to answer:
    Question:
    {text}
    Options:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    Answer:'''

    data.at[i, 'user_content'] = prompt

data2 = data
data = data[['system_message','user_content']]

In [ ]:
data.to_excel('QA-Math-CoT-gpt.xlsx')

In [ ]:
# Convert to JSON String
data = data.to_json(orient='records')

# parse JSON string and convert it into a list of python dictionaries
datadict = json.loads(data)

## Format Prompt

In [ ]:
# format dict into shape for fine tuning
CoT_data = format_prompt(datadict)

# Save as jsonl
convert_to_jsonl_and_save(CoT_data, 'SA-CoT.jsonl')

[{'messages': [{'role': 'system', 'content': 'Your task is to select and write only the correct answer from the options (a, b, c, d).\nTo answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:\na/b/c/d) Answer\n\nFollow these reasoning steps to determine the correct answer:\n\nStep 1: Understand the question\nCarefully read and comprehend the question. Identify what is being asked, including the underlying need and the context.\n\nStep 2: Analyze the options\nCarefully read each of the four options. Consider the meaning of each option and how it relates to the question.\n\nStep 3: Evaluate the options\nApply your knowledge and reasoning to assess each option. Eliminate clearly incorrect options. Compare the remaining options to determine which is most accurate.\n\nStep 4: Select the best answer\nBased on your evaluation in the previous steps, choose the single most logical and correct option.\n\nStep 5: 

## Predict

In [ ]:
response = client.chat.completions.create(
      model = model,
      messages= CoT_data[99]['messages'],
      temperature = 0.2,
      max_tokens= 512
  )

response.choices[0].message.content

'c) 9'

In [ ]:
def predict(test, model):
  response = client.chat.completions.create(
      model = model,
      messages= test,
      temperature = 0.2,
      max_tokens= 512
  )
  return response.choices[0].message.content

In [ ]:
def store_predictions(test_df, model, test_data):
  print("fine tuned model id is :", model)
  test_df['Prediction']= None

  for index, row in test_df.iterrows():
    test_message = test_data[index]['messages']
    prediction_result = predict(test_message, model)
    test_df.at[index, 'Prediction'] = prediction_result

  test_df.to_csv("QA-Math-GPT41-CoT-predictions-temp0.2.csv")

In [ ]:
y_true = data2['Answer Key'].values

In [ ]:
cot = pd.DataFrame()

cot['Question'] = data2['Question']
cot['Label'] = y_true
cot = cot.reset_index(drop=True)
cot.head()

,Question,Label
0,احد الاعداد التالية يقبل القسمة على 6 دون باق ...,A
1,جد ناتج مايلي : 6 × 3981\n,C
2,عدد من منزلتین مجموعهما = ١٠,A
3,من أدوات القياس,C
4,أي من المسائل التالية ناتج جمعها يساوي 16,A


In [ ]:
store_predictions(cot, model, CoT_data)

fine tuned model id is : gpt-4.1-2025-04-14


In [ ]:
pred = pd.read_csv('QA-Math-GPT41-CoT-predictions-temp0.2.csv')

In [ ]:
pred['Prediction'].value_counts()

,count
Prediction,
d) nan,4
b) 2,4
a) 12,3
a) 7,2
a) 9,2
...,...
d) 878,1
a) وتر,1
c) 1,1


In [ ]:
pred['Prediction'].isnull().sum()

np.int64(0)

In [ ]:
import numpy as np

preds = []
for answer in pred['Prediction']:
  if "a)" in answer:
    preds.append("A")
  elif "b)" in answer:
    preds.append("B")
  elif "c)" in answer:
    preds.append("C")
  elif "d)" in answer:
    preds.append("D")
  else:
    preds.append("Unclassified")

In [ ]:
np.unique(preds)

array(['A', 'B', 'C', 'D'], dtype='<U1')

In [ ]:
from sklearn.metrics import (f1_score,
                             precision_score,
                             recall_score,
                             classification_report,
                             confusion_matrix)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

labels = ['A', 'B', 'C', 'D']
mapping = {'A':0, 'B':1, 'C':2, 'D':3}

def map_func(x):
    return mapping.get(x, 1)

y_true = np.vectorize(map_func)(cot.Label)
y_pred = np.vectorize(map_func)(preds)

# Calculate accuracy
accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f'Accuracy: {accuracy:.3f}')

# Generate accuracy report
unique_labels = set(y_true)  # Get unique labels

f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\nf1_score: ', f1t)

prec = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\precision: ', prec)

recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print('\recall: ', recall)

# Generate classification report
class_report = classification_report(y_true=y_true,
                                     y_pred=y_pred,
                                     digits = 4,
                                     target_names=labels)
print('\nClassification Report:')
print(class_report)

Accuracy: 0.795

f1_score:  0.7949358353510896
\precision:  0.8085226929932812
ecall:  0.795

Classification Report:
              precision    recall  f1-score   support

           A     0.7600    0.8769    0.8143        65
           B     0.7692    0.8333    0.8000        48
           C     0.9216    0.7015    0.7966        67
           D     0.6818    0.7500    0.7143        20

    accuracy                         0.7950       200
   macro avg     0.7832    0.7904    0.7813       200
weighted avg     0.8085    0.7950    0.7949       200

